In [1]:
from scipy.optimize import brentq
import numpy as np
from datetime import datetime
import time, sys, os, pyvisa, subprocess
import matplotlib as mpl
import matplotlib.pyplot as plt
import time

In [2]:
hp34461a = 'GPIB0::22::INSTR'
keithley2450_gpib = 'GPIB0::18::INSTR'
port ='S21'

In [3]:
# use Python 32bit for PicoVNA108

try:
    # __file__ is not defined in Jupyter
    current_directory = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Use the current working directory in Jupyter
    current_directory = os.getcwd()

main_directory = os.path.dirname(current_directory) 
if main_directory not in sys.path:
    sys.path.append(main_directory)
from Instrument_Drivers.PicoVNA108 import get_picoVNA_smith
from Instrument_Drivers.Instrument_dict import *
rm = pyvisa.ResourceManager()

# subprocess.call(['sh','SD_pywin32error.sh'])
# this command delete r'C:\Users\<username>\AppData\Local\Temp\2\gen_py' to solve a potential pywin32 error that usually happens after you use the PICOVNA 3 program
global msmt_flag, my_note
msmt_flag = None
my_note = ''

def my_form(kwargs):
    if 'smith' in kwargs.keys():
        lens = len(kwargs['smith'].freqs)
    else:
        lens = 1
    list = []
    dataToSave = []
    for key in kwargs:
        if key=='smith':
            smith = kwargs[key]
            dataToSave += [smith.freqs]
            dataToSave += [smith.log_mag]
            dataToSave += [smith.phase_rad]
            dataToSave += [smith.real]
            dataToSave += [smith.imag]
            list += ['VNA_freqs', 'VNA_log_mag', 'VNA_phase_rad', 'VNA_real', 'VNA_imag']
        else:
            dataToSave += [np.full(lens, kwargs[key])]
            list += [key]
    data = np.column_stack(dataToSave)
    axis = ''
    for x in list:
        axis += f"{x}_{order}\t\t\t"
    return data, axis


'''---------------------Run funcs---------------------'''
def get_sweep(start,stop,step_size):
    num_steps = int((abs(float(start) - float(stop)) / float(step_size))) + 1
    return np.linspace(float(start), float(stop), num_steps)

def run_single(sweep,order,f_min,f_max,average=250,power=-5,name=None,number_of_points=1001,bandwidth=1000):
    global my_note
    set(sweep)
    value = read()
    avg = f"\n average for {average} times"
    print(f"{datetime.now().strftime('%Y.%m.%d')}", " ", f"{datetime.now().strftime('%H:%M:%S')}", "  ",
          order, " started")
    msg = 'VNA data for'
    for key, item in value.items():
        msg += f" {key}={item}, "
    vs = get_picoVNA_smith(port=port, f_min=f_min, f_max=f_max, number_of_points=number_of_points, power=power, bandwidth=bandwidth, Average=average)
    print(msg+' recorded')
    value.update({'smith':vs})
    data, axis = my_form(value)
    if name is not None:
        file_name = f"{name}.{order}"
    else:
        file_name = f"{title}.{order}"
    file_real_path = data_dir + '\\' + datetime.now().strftime('%Y%m%d') + "\\" + title + "\\" + file_name
    while os.path.exists(file_real_path):
        order = order + 1
        file_name = ''.join(file_name.split('.')[:-1]) + f'.{order}'
        file_real_path = data_dir + '\\' + datetime.now().strftime('%Y%m%d') + "\\" + title + "\\" + file_name
    os.makedirs(data_dir + '\\' + datetime.now().strftime('%Y%m%d') + "\\" +title, exist_ok=True)
    my_note += avg + ', power=' + f'{power}' + ', bandwidth=' + f'{bandwidth}' + '\n'
    np.savetxt(file_real_path, data, delimiter='\t',
               header=f"{datetime.now().strftime('%Y%m%d')}" + " " + f"{datetime.now().strftime('%H%M%S')}" + '\n' + \
                      my_note + f"{axis}")

def dry_sweep(start, stop, step_size=0.01, delay=0.9):
    print(f"{datetime.now().strftime('%Y.%m.%d')}", " ", f"{datetime.now().strftime('%H:%M:%S')} ", 'Sweep started from:')
    read()
    for sweep in get_sweep(start=start, stop=stop, step_size=step_size):
        sweep = round(sweep,ndigits=4)
        set(sweep,delay=delay)
        value = read(printable=False)
        data, axis = my_form(value)
        os.makedirs(data_dir + '\\' + datetime.now().strftime('%Y%m%d') + "\\" +title, exist_ok=True)
        dry_sweep_real_path = data_dir + '\\' + datetime.now().strftime('%Y%m%d') + "\\" + title + "\\" + title + '_dry_sweep'
        if not os.path.exists(dry_sweep_real_path):
            np.savetxt(dry_sweep_real_path,data,delimiter='\t',
                       header=f"{datetime.now().strftime('%Y.%m.%d')}" + " " + f"{datetime.now().strftime('%H:%M:%S')}" +
                              '\n' + f"{axis}")
        else:
            with open(dry_sweep_real_path, "ab") as f:
                np.savetxt(f, data, delimiter='\t')
    print(f"{datetime.now().strftime('%Y.%m.%d')}", " ", f"{datetime.now().strftime('%H:%M:%S')} ", 'Sweep ended at :')
    read()
    return sweep

# def wet_sweep(start, stop, step_size, order, last_v, f_min, f_max, number_of_points=1001, power=-5, average=250, dry_step_size=0.01, dry_delay=0.9, wet_delay =0.01, duplicate=1):
#     for sweep in get_sweep(start=start, stop=stop, step_size=step_size):
#         last_v = dry_sweep(last_v, sweep, step_size=dry_step_size, delay=dry_delay)
#         time.sleep(wet_delay)
#         for i in range(duplicate):
#             order += 1
#             run_single(sweep, order, f_min=f_min, f_max=f_max, average=average, power=power, number_of_points=number_of_points)
#             last_v = sweep
#     return last_v

def wait_for_stable_resistance(timeout=180, check_interval=1,machineGPIB=hp34461a):
    """
    Wait until the resistance value remains unchanged for the specified timeout.
    
    :param timeout: The time in seconds for which the resistance value must remain unchanged.
    :param check_interval: The interval in seconds at which to check the resistance value.
    """
    start_time = time.time()
    initial_value = round(hp34461a_get_ohm_4pt(machineGPIB),2)
    
    while True:
        time.sleep(check_interval)
        current_value = round(hp34461a_get_ohm_4pt(machineGPIB),2)
        
        if current_value != initial_value:
            start_time = time.time()
            initial_value = current_value
        
        elapsed_time = time.time() - start_time
        
        if elapsed_time >= timeout:
            break

def wet_sweep_YS(start, stop, step_size, order, last_v, f_min, f_max, number_of_points=1001, power=-5, average=250, dry_step_size=0.01,  wet_delay=100, dry_delay=0.9, times=1):
    for sweep in get_sweep(start=start, stop=stop, step_size=step_size):
        last_v = dry_sweep(last_v, sweep, step_size=dry_step_size, delay=dry_delay)
    
        wait_for_stable_resistance(timeout=wet_delay, check_interval=1,machineGPIB=hp34461a)

        for i in np.arange(0,times):
            order += 1
            run_single(sweep, order, f_min=f_min, f_max=f_max, average=average, power=power, number_of_points=number_of_points)
            last_v = sweep
    return last_v

'''---------------------Start your sequence here---------------------'''
# define the program for each sweep
def set(value, delay=0.9):
    global msmt_flag
    if value is not None:
        if msmt_flag =='Heater sweep':
            keithley2450_set_sour_voltage_V(keithley2450_gpib, value)
        time.sleep(delay)
    time.sleep(0.1)

def read(printable=True,*arg):
    global msmt_flag
    read = {}
    read.update({'timestamp': time.time()})
    if msmt_flag =='Heater sweep':
        read.update({'V_Heater':keithley2450_get_sour_voltage_V(keithley2450_gpib)})
        read.update({'I_Heater':keithley2450_get_meas_currrent_A(keithley2450_gpib)})
        read.update({'R_RuOx': hp34461a_get_ohm_4pt(hp34461a)})
    if printable:
        msg = ''
        for key, item in read.items():
            msg += f'{key}={item}, '
        print(msg)
    return read

def R2T_900(R):  # from 6K to 96K
    a0, a1, a2, a3, a4, a5, a6, a7, a8 = 6.75543700e+03, -1.59063022e+04, 1.92990842e+04, -1.30574569e+04, 5.39085191e+03, -1.39623593e+03, 2.22191016e+02, -1.99044732e+01, 7.69758689e-01
    
    if np.isscalar(R):
        R = np.array([R])

    if np.any(R < 946.97) or np.any(R > 1135.68):
        print('out of range: results may not be accurate')

    def f(T, R):
        log_T = np.log(T)
        return (a0 + a1 * log_T + a2 * log_T**2 + a3 * log_T**3 +
                a4 * log_T**4 + a5 * log_T**5 + a6 * log_T**6 +
                a7 * log_T**7 + a8 * log_T**8) - R

    def find_temp(R):
        try:
            return brentq(f, 3, 100, args=(R,))
        except ValueError:
            raise ValueError('root finding did not converge within the temperature range 6K to 96K')

    temperatures = np.array([find_temp(R_input) for R_input in R])
    
    if temperatures.size == 1:
        return temperatures[0]
    else:
        return temperatures

def is_temp_stable(check_interval=3,safety_duration=300,tol=3):
    Ri = round(hp34461a_get_ohm_4pt(hp34461a),tol)
    time.sleep(check_interval)
    Rf = round(R2T_900(hp34461a_get_ohm_4pt(hp34461a)),tol)
    judgement=Rf==Ri
    if judgement==False:
        pass 
    else:
            Ri_ = round(hp34461a_get_ohm_4pt(hp34461a),2)
            time.sleep(safety_duration)
            Rf_ = round(hp34461a_get_ohm_4pt(hp34461a),2)
            judgement=Ri_==Rf_
    return judgement

def is_temp_lowest():
    if R2T_900(hp34461a_get_ohm_4pt(hp34461a)) < 18:
        return True
    else:
        return False 
    
def is_temp_above(threshold=96):
    if R2T_900(hp34461a_get_ohm_4pt(hp34461a)) > threshold:
        return True
    else:
        return False 

In [4]:
# '''DC sweep Gate'''
# msmt_flag ='Heater sweep'
# data_dir = r'C:\Users\Crow108\Documents\Data\YS\AFMR_YBCO_A_Q3KCP30_ICET\HeatingUp'
# my_note = "202407302225_Yinyao_IceT_HeatingSweep"
# last_v = 0
# order = 0
# title = "20240731" #some unique feature you want to add in title
# last_v = wet_sweep_YS(start=last_v,
#                    stop=50,
#                    step_size=5,
#                    order=order,
#                    last_v=last_v,
#                    f_min=6960,
#                    f_max=7060,
#                    power=-10,
#                    average=5000,
#                    wet_delay=5,
#                    dry_step_size=1,
#                    times=1,
#                    dry_delay=5)
# print('done')

In [5]:
print(round(R2T_900(1071.6),3))
# print(round(hp34461a_get_ohm_4pt(hp34461a),3))

14.731


In [250]:
'''Take trace_manual'''
msmt_flag ='Heater sweep'
data_dir = r'C:\Users\Crow108\Documents\Data\YS\AFMR_YBCO_A_Q3KCP30_ICET\Heating'
order = 0
# temp=round(R2T_900(hp34461a_get_ohm_4pt(hp34461a)),2)
title = "2" # some unique feature you want to add in title
my_note = ""

while True:
    run_single(sweep=None, order=order, f_min=6300, f_max=7100, average=10000, number_of_points=2001, power=-10)
    time.sleep(20)
    if is_temp_above():
        print("Exiting loop. ","Temp=",round(R2T_900(hp34461a_get_ohm_4pt(hp34461a)),3))
        break


timestamp=1722916029.5100718, V_Heater=21.0, I_Heater=-1.763884e-09, R_RuOx=1055.9952, 
2024.08.05   22:47:09    0  started
VNA data for timestamp=1722916029.5100718,  V_Heater=21.0,  I_Heater=-1.763884e-09,  R_RuOx=1055.9952,  recorded
timestamp=1722916057.9852805, V_Heater=21.0, I_Heater=-2.019138e-09, R_RuOx=1055.5198, 
2024.08.05   22:47:38    0  started
VNA data for timestamp=1722916057.9852805,  V_Heater=21.0,  I_Heater=-2.019138e-09,  R_RuOx=1055.5198,  recorded
timestamp=1722916086.0926335, V_Heater=21.0, I_Heater=-2.427541e-09, R_RuOx=1055.0316, 
2024.08.05   22:48:06    0  started
VNA data for timestamp=1722916086.0926335,  V_Heater=21.0,  I_Heater=-2.427541e-09,  R_RuOx=1055.0316,  recorded
timestamp=1722916114.5098457, V_Heater=21.0, I_Heater=-1.516427e-09, R_RuOx=1054.5423, 
2024.08.05   22:48:35    0  started
VNA data for timestamp=1722916114.5098457,  V_Heater=21.0,  I_Heater=-1.516427e-09,  R_RuOx=1054.5423,  recorded
timestamp=1722916142.7164164, V_Heater=21.0, I_Heate

ValueError: root finding did not converge within the temperature range 6K to 96K

In [245]:
'''Take trace_manual'''
msmt_flag ='Heater sweep'
data_dir = r'C:\Users\Crow108\Documents\Data\YS\AFMR_YBCO_A_Q3KCP30_ICET\Cooling'
order = 0
temp=round(R2T_900(hp34461a_get_ohm_4pt(hp34461a)),2)
title = str(temp) # some unique feature you want to add in title
my_note = ""
run_single(sweep=None, order=order, f_min=6300, f_max=7100, average=10000, number_of_points=2001, power=-10)


timestamp=1722915807.978259, V_Heater=21.0, I_Heater=-1.796699e-09, R_RuOx=1056.429, 
2024.08.05   22:43:28    0  started
VNA data for timestamp=1722915807.978259,  V_Heater=21.0,  I_Heater=-1.796699e-09,  R_RuOx=1056.429,  recorded
